# Notebook 5 — Etiquetado de imágenes

**Curso:** EITC/AI/GVAPI — European IT Certification, Google Vision API
**Lección 5:** Labelling Images (Labels Detection)

In [ ]:
import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "ruta/a/tu/clave.json"

from google.cloud import vision
client = vision.ImageAnnotatorClient()
print("Cliente listo.")


## Detección de etiquetas (`label_detection`)

`label_detection` identifica objetos, escenas y hasta conceptos abstractos (por ejemplo, "felicidad") presentes en una imagen, y devuelve una lista de etiquetas con su puntuación de confianza.

**Campos de la respuesta:**

- `description`: texto de la etiqueta (por ejemplo, "cat")
- `score`: puntuación de confianza
- `topicality`: importancia de la etiqueta dentro de la imagen

> **Detalle a comprobar tú mismo**: en la versión de la API descrita en el material del curso, `score` y `topicality` devolvían el mismo valor — un comportamiento documentado que Google puede haber corregido ya. Compruébalo con tus propias imágenes en el ejercicio 1.

**Requisitos y límites**: formatos JPEG, PNG o GIF, con un tamaño máximo de cuatro megapíxeles.

**Errores habituales**: fallos de autenticación, problemas de red, cuota de API agotada, imagen corrupta o en un formato no soportado, y parámetros incorrectos en la llamada. Envuelve siempre tus llamadas en un bloque `try/except` (lo practicarás en el ejercicio 3).

In [ ]:
def extract_labels(image_path):
    with open(image_path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)
    response = client.label_detection(image=image)
    labels = response.label_annotations

    return [(label.description, label.score, label.topicality) for label in labels]


# print(extract_labels("gato.jpg"))


## Ejercicio 1 — Etiquetas en un DataFrame de pandas

Completa `etiquetas_en_dataframe` para que llame a `extract_labels` y construya un DataFrame de pandas con columnas `description`, `score` y `topicality`. Aprovecha para comprobar si `score` y `topicality` siguen coincidiendo en tu propia cuenta de Google Cloud.

In [ ]:
import pandas as pd

def etiquetas_en_dataframe(image_path):
    etiquetas = extract_labels(image_path)

    # TODO: construye el DataFrame con columnas description, score y topicality
    df = ...
    return df


# df = etiquetas_en_dataframe("gato.jpg")
# df


<details>
<summary><strong>Ver solución</strong></summary>

```python
import pandas as pd

def etiquetas_en_dataframe(image_path):
    etiquetas = extract_labels(image_path)
    df = pd.DataFrame(etiquetas, columns=["description", "score", "topicality"])
    return df


df = etiquetas_en_dataframe("gato.jpg")
df
```

</details>

## Ejercicio 2 — Filtrar por umbral de confianza

Completa `etiquetas_fiables` para que devuelva solo las etiquetas cuyo `score` supere un umbral dado (por defecto 0.85). Pruébala con imágenes de distinta calidad (nítida, borrosa, con oclusión parcial) y compara cuántas etiquetas superan el umbral en cada caso.

In [ ]:
def etiquetas_fiables(image_path, umbral=0.85):
    etiquetas = extract_labels(image_path)

    # TODO: filtra la lista de tuplas (description, score, topicality) por score > umbral
    fiables = ...
    return fiables


# print(etiquetas_fiables("gato_borroso.jpg", umbral=0.85))


<details>
<summary><strong>Ver solución</strong></summary>

```python
def etiquetas_fiables(image_path, umbral=0.85):
    etiquetas = extract_labels(image_path)
    fiables = [e for e in etiquetas if e[1] > umbral]
    return fiables


print(etiquetas_fiables("gato_borroso.jpg", umbral=0.85))
```

</details>

## Ejercicio 3 — Manejo de errores en el pipeline de etiquetado

Completa `extract_labels_seguro` para que envuelva la llamada a `label_detection` en un bloque `try/except`, capturando al menos dos escenarios de error (por ejemplo, ruta de archivo inexistente y fallo de autenticación), y que devuelva un mensaje de error significativo en lugar de lanzar una excepción sin control.

In [ ]:
def extract_labels_seguro(image_path):
    # TODO 1: envuelve la apertura del archivo y la llamada a la API en un bloque try/except
    try:
        ...
    except FileNotFoundError:
        return {"error": f"No se encontró el archivo: {image_path}"}
    except Exception as e:
        return {"error": f"Error al llamar a la API: {e}"}


# print(extract_labels_seguro("archivo_que_no_existe.jpg"))


<details>
<summary><strong>Ver solución</strong></summary>

```python
def extract_labels_seguro(image_path):
    try:
        with open(image_path, "rb") as image_file:
            content = image_file.read()

        image = vision.Image(content=content)
        response = client.label_detection(image=image)
        return [(label.description, label.score) for label in response.label_annotations]
    except FileNotFoundError:
        return {"error": f"No se encontró el archivo: {image_path}"}
    except Exception as e:
        return {"error": f"Error al llamar a la API: {e}"}


print(extract_labels_seguro("archivo_que_no_existe.jpg"))
```

</details>

---

**Siguiente paso:** abre `06_Comprension_Avanzada.ipynb` para trabajar con monumentos, logotipos, objetos y Safe Search.